In [ ]:
from sfnb_setup import setup_notebook
result = setup_notebook(config="julia_smoke_test_config.yaml")

## 1. Julia Environment Setup

Initialize JuliaCall and the `%%julia` cell magic.


In [ ]:
from julia_helpers import setup_julia_environment
setup_julia_environment()

## 2. Julia Execution

Validate basic Julia language features: variables, string interpolation, math.


In [ ]:
%%julia
greeting = "Hello from Julia"
println(greeting)

numbers = [1, 2, 3, 4, 5]
doubled = numbers .* 2
println("Doubled: $doubled")

total = sum(numbers)
println("Sum of 1..5 = $total")

if total == 15 && doubled == [2, 4, 6, 8, 10]
    println("[PASS] Julia execution")
else
    println("[FAIL] Julia execution")
end

## 3. DataFrames

Validate that the DataFrames.jl package loads and works correctly.


In [ ]:
%%julia
using DataFrames

df = DataFrame(
    name = ["Alice", "Bob", "Charlie"],
    score = [95, 87, 92],
    grade = ["A", "B+", "A-"]
)
println(df)
println()

avg_score = round(mean(df.score), digits=1)
println("Average score: $avg_score")

if nrow(df) == 3 && ncol(df) == 3
    println("[PASS] DataFrames")
else
    println("[FAIL] DataFrames")
end

## 4. Variable Transfer (Python to Julia)

Pass data from Python into Julia using the `-i` flag.


In [ ]:
py_data = {"languages": ["Julia", "Python", "Scala"], "scores": [95, 90, 88]}
print(f"Created Python dict with {len(py_data['languages'])} items")

In [ ]:
%%julia -i py_data
println("Received from Python:")
println("  Languages: $(py_data["languages"])")
println("  Scores: $(py_data["scores"])")

if length(py_data["languages"]) == 3
    println("[PASS] Python-to-Julia transfer")
else
    println("[FAIL] Python-to-Julia transfer")
end

## 5. Variable Transfer (Julia to Python)

Pass data from Julia back to Python using the `-o` flag.


In [ ]:
%%julia -o jl_result
jl_result = Dict(
    "julia_version" => string(VERSION),
    "dataframes_ok" => true,
    "test_value" => 42
)
println("Created Julia Dict for transfer: $jl_result")

In [ ]:
if jl_result is not None and jl_result.get("test_value") == 42:
    print(f"Received from Julia:")
    for k, v in jl_result.items():
        print(f"  {k}: {v}")
    print("[PASS] Julia-to-Python transfer")
else:
    print("[FAIL] Julia-to-Python transfer")

## Summary


In [ ]:
%%julia
println()
println("=" ^ 60)
println("  Julia Smoke Test -- Summary")
println("=" ^ 60)
println("  Julia version:  $(VERSION)")
println("-" ^ 60)
println("  Check output above for [PASS] / [FAIL] per section")
println("=" ^ 60)

In [ ]:
# CI results -- writes a results table when run non-interactively
try:
    from snowflake.snowpark.context import get_active_session
    _ci_session = get_active_session()
    _ci_session.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.JULIA_SMOKE_TEST_RESULTS AS
        SELECT 'julia_smoke_test' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI results written to NOTEBOOK_CI.JULIA_SMOKE_TEST_RESULTS")
except Exception:
    pass